# Lesson 15 Lab — Registers, Warps, and Occupancy Trade-offs

**Puzzle:** When BLOCK, num_warps, live values, and latency change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates BLOCK, num_warps, live values, and latency and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

BLOCK and num_warps influence work per program, parallel issue, live values, and the number of resident programs. Timing a controlled configuration sweep reveals sensitivity, but it does not directly report registers or achieved occupancy.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["BLOCK, num_warps, live values, and latency"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Maximizing theoretical occupancy can reduce performance when it forces smaller tiles or more redundant traffic.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 15
LESSON_TITLE = 'Registers, Warps, and Occupancy Trade-offs'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260828
}


## 5. Freeze the experiment

**Experiment:** Sweep four BLOCK/warp pairs for the same affine kernel and retain all latency samples.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.01696000061929226,
  "secondary": 1.7037735366237647,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "best_config": "b1024-w8",
    "worst_config": "b128-w4",
    "sweep": {
      "b128-w4": {
        "median_ms": 0.028896000236272812,
        "samples_ms": [
          0.036288000643253326,
          0.03161599859595299,
          0.0306560005992651,
          0.028063999488949776,
          0.02800000086426735,
          0.029440000653266907,
          0.030400000512599945,
          0.02940800040960312,
          0.028896000236272812,
          0.028960000723600388,
          0.02844800055027008,
          0.027488000690937042,
          0.028511999174952507,
          0.027103999629616737,
          0.026655999943614006
        ]
      },
      "b256-w4": {
        "median_ms": 0.02006400004029274,
        "samples_ms": [
          0.025855999439954758,
          0.024288000538945198,
          0.021536000072956085,
          0.0207359

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Best median | 0.0170 ms |
| Worst/best ratio | 1.704x |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The best measured configuration was b1024-w8; the slowest/best latency ratio was 1.70x. Occupancy was not inferred from timing alone.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use profiler resource fields to explain a timing curve; do not infer occupancy from the fastest point alone.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 15,
  "title": "Registers, Warps, and Occupancy Trade-offs",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260828
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.01696000061929226,
    "secondary": 1.7037735366237647,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "best_config": "b1024-w8",
      "worst_config": "b128-w4",
      "sweep": {
        "b128-w4": {
          "median_ms": 0.028896000236272812,
          "samples_ms": [
            0.036288000643253326,
            0.03161599859595299,
            0.0306560005992651,
            0.028063999488949776,
            0.02800000086426735,
            0.029440000653266907,
            0.030400000512599945,
            0.0

## 10. Make the bounded decision

> Use profiler resource fields to explain a timing curve; do not infer occupancy from the fastest point alone.

**Failure analysis:** Maximizing theoretical occupancy can reduce performance when it forces smaller tiles or more redundant traffic.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
